# Figuring out BCPNN

In [6]:
from vigipy import *
import pandas as pd

import duckdb
import sys
sys.path.insert(0, "..")  # per trovare il modulo vigipy locale

Overall structure of the data:

In [7]:
from pyarrow.parquet import ParquetFile
import pyarrow as pa 

n=100000

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat_deduped.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = n)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 

In [ ]:
[name for name in ae_df.columns]

['safetyreportid',
 'receivedate',
 'receive_year',
 'receive_quarter',
 'serious',
 'outcome_death',
 'outcome_lifethreat',
 'outcome_hosp',
 'outcome_disab',
 'reporter_country',
 'reporter_qual',
 'age_years',
 'age_stratum',
 'sex',
 'drug_name',
 'drug_name_source',
 'drug_characterization',
 'drug_indication',
 'reaction_pt']

: 

import sys
sys.path.insert(0,'..')

In [ ]:
from vigipy import GPS
from vigipy.utils import Container
from src.contingency_table import build_contingency_table, qc_contingency_table

ct = pd.read_parquet("data/contingency_table.parquet")

def contingency_to_vigipy(ct: pd.DataFrame) -> tuple[Container, int]:
    """
    Converte la contingency table 2x2 nel formato atteso da vigipy GPS.
    """
    df = ct.copy()
    df = df.rename(columns={
        "drug": "product_name",
        "pt":   "ae_name",
        "a":    "events",
    })
    df["product_aes"]          = df["events"] + df["b"]   # a + b
    df["count_across_brands"]  = df["events"] + df["c"]   # a + c

    N = int(df["n"].iloc[0])  # totale unico per tutto il sottoinsieme

    container = Container(params=False)
    container.data = df[["product_name", "ae_name", "events",
                          "product_aes", "count_across_brands"]]
    container.N    = N

    # Matrice di contingenza pivot (drug x PT) — serve per stimare i prior
    container.contingency = df.pivot_table(
        index="product_name",
        columns="ae_name",
        values="events",
        fill_value=0
    )

    return container, N

container, N = contingency_to_vigipy(ct)
print(f"N totale report: {N}")
print(f"Coppie drug-PT : {len(container.data)}")
container.data.head()


N totale report: 95990
Coppie drug-PT : 7


,product_name,ae_name,events,product_aes,count_across_brands
0,LAPATINIB,DIARRHOEA,7,45,3283
1,LAPATINIB,FEBRILE NEUTROPENIA,4,45,441
2,LAPATINIB,HYPOCALCAEMIA,4,45,106
3,LAPATINIB,DISEASE PROGRESSION,3,45,684
4,LAPATINIB,NAUSEA,3,45,4744


Funziona! caccio dentro bcpnn, the article on Iapatinib sets IC>0, lower limit of CI>0, N>=3

In [3]:
res=bcpnn(container=container, min_events=3, decision_metric="rank", ranking_statistic="quantile")
# figure out cosa devo fare per fare quello che hanno fatto per iapatinib

In [4]:
res.all_signals

# bisogna trovare i segnali rilevanti quali sono 


,Product,Adverse Event,Count,Expected Count,quantile,count/expected,product margin,event margin,fdr,FNR,Se,Sp
2,LAPATINIB,HYPOCALCAEMIA,4.0,0.049693,0.701865,80.494759,45.0,106.0,0.002825,1.127979e+00,0.473795,0.711458
1,LAPATINIB,FEBRILE NEUTROPENIA,4.0,0.206740,0.511914,19.347947,45.0,441.0,0.003384,1.094459e+00,0.315686,0.769213
0,LAPATINIB,DIARRHOEA,7.0,1.539067,0.445426,4.548211,45.0,3283.0,0.003042,1.052328e+00,0.157897,1.000000
3,LAPATINIB,DISEASE PROGRESSION,3.0,0.320658,-0.147881,9.355750,45.0,684.0,0.011605,1.317860e+00,0.626164,0.067664
5,LAPATINIB,CHILLS,3.0,0.335191,-0.163810,8.950117,45.0,715.0,0.087951,1.993233e+00,0.866697,0.009864
6,LAPATINIB,PYREXIA,3.0,1.017293,-0.765320,2.949002,45.0,2170.0,0.098005,9.969577e+06,1.000000,0.004435
4,LAPATINIB,NAUSEA,3.0,2.223982,-1.447009,1.348932,45.0,4744.0,0.097617,1.495763e+00,0.714593,0.012352
